# Assignment 5 — Catching Data Leakage Before It Catches You
Feature Engineering & MLOps · Data Leakage: Target Leakage & Preprocessing Leakage

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42

df = pd.read_csv('../data/raw/Unit-02/customer_churn_a5.csv')
print(df.shape)
df.head()

(600, 8)


,customer_id,tenure_months,monthly_charges,contract_type,support_calls,churn,days_since_cancellation,final_bill_amount
0,20406,13,99.17,Month-to-month,2,0,NaN,NaN
1,20531,24,66.77,Month-to-month,1,0,NaN,NaN
2,20307,15,44.03,One year,1,0,NaN,NaN
3,20391,3,61.08,Two year,3,0,NaN,NaN
4,20147,8,106.72,Two year,1,0,NaN,NaN


## Section 4.1 — Target leakage: prove it, don't just assert it

In [2]:
# Step 1: confirm days_since_cancellation is only present when churn=1
n_non_null_cancel = df['days_since_cancellation'].notna().sum()
n_churn = (df['churn'] == 1).sum()
print(f'Non-null days_since_cancellation: {n_non_null_cancel}')
print(f'churn=1 rows:                    {n_churn}')
print(f'Exact match: {n_non_null_cancel == n_churn}')

Non-null days_since_cancellation: 149
churn=1 rows:                    149
Exact match: True


In [3]:
# Step 2: correlation of leaked vs legitimate features against churn
corr_final_bill = df['final_bill_amount'].fillna(-1).corr(df['churn'])
corr_tenure = df['tenure_months'].corr(df['churn'])
print(f'Correlation(final_bill_amount [filled -1], churn): {corr_final_bill:.4f}')
print(f'Correlation(tenure_months, churn):                 {corr_tenure:.4f}')

Correlation(final_bill_amount [filled -1], churn): 0.9250
Correlation(tenure_months, churn):                 -0.2207


In [4]:
# Setup: same train/test split for Models A and B (split first — no preprocessing leakage here)
legit_features = ['tenure_months', 'monthly_charges', 'contract_type', 'support_calls']
leaked_features = ['days_since_cancellation', 'final_bill_amount']

X = df[legit_features + leaked_features].copy()
X[leaked_features] = X[leaked_features].fillna(-1)  # constant fill — no fitted statistic, safe before split
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

def evaluate_model(X_tr, X_te, num_cols, cat_cols):
    scaler = StandardScaler()
    X_tr_num = scaler.fit_transform(X_tr[num_cols])
    X_te_num = scaler.transform(X_te[num_cols])
    X_tr_cat = pd.get_dummies(X_tr[cat_cols], columns=cat_cols, drop_first=False)
    X_te_cat = pd.get_dummies(X_te[cat_cols], columns=cat_cols, drop_first=False)
    X_te_cat = X_te_cat.reindex(columns=X_tr_cat.columns, fill_value=0)
    X_tr_final = np.hstack([X_tr_num, X_tr_cat.astype(float).values])
    X_te_final = np.hstack([X_te_num, X_te_cat.astype(float).values])
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    model.fit(X_tr_final, y_train)
    y_pred = model.predict(X_te_final)
    y_proba = model.predict_proba(X_te_final)[:, 1]
    return accuracy_score(y_test, y_pred), roc_auc_score(y_test, y_proba)

In [5]:
# Step 3: Model A — legitimate features only
num_a = ['tenure_months', 'monthly_charges', 'support_calls']
cat_a = ['contract_type']
acc_a, auc_a = evaluate_model(X_train, X_test, num_a, cat_a)
print(f'Model A  accuracy: {acc_a:.4f}   ROC-AUC: {auc_a:.4f}')

Model A  accuracy: 0.7533   ROC-AUC: 0.7783


In [6]:
# Step 4: Model B — legitimate + leaked features (missing already filled with -1)
num_b = ['tenure_months', 'monthly_charges', 'support_calls', 'days_since_cancellation', 'final_bill_amount']
cat_b = ['contract_type']
acc_b, auc_b = evaluate_model(X_train, X_test, num_b, cat_b)
print(f'Model B  accuracy: {acc_b:.4f}   ROC-AUC: {auc_b:.4f}')
print()
print(f'Accuracy gap (B - A): {acc_b - acc_a:+.4f}')
print(f'ROC-AUC gap  (B - A): {auc_b - auc_a:+.4f}')

Model B  accuracy: 1.0000   ROC-AUC: 1.0000

Accuracy gap (B - A): +0.2467
ROC-AUC gap  (B - A): +0.2217


**Step 5 — What would happen if Model B were deployed on brand-new applicants?**

New applicants who haven't churned yet have no value for `days_since_cancellation` or `final_bill_amount` — these fields are *consequences* of churning, not predictors of it. Model B has learned a near-perfect shortcut (e.g., "fill = -1 means not churned; any other value means churned"), so in production it would either crash on missing columns or, if fed default/-1 values, confidently label every new applicant as non-churn with no real predictive power. The inflated validation score was measuring the model's ability to read the answer key, not to predict behaviour — real-world performance would collapse to (or below) chance.

## Section 4.2 — Preprocessing leakage: the wrong order vs. the right order

In [7]:
# Legitimate numeric features only (no contract_type here, per the task)
numeric_cols = ['tenure_months', 'monthly_charges', 'support_calls']
X_num = df[numeric_cols]
y_all = df['churn']

# Step 1 (WRONG order): fit scaler on ENTIRE dataset, THEN split
scaler_wrong = StandardScaler()
scaler_wrong.fit(X_num)  # saw test data too — preprocessing leakage
print('WRONG order — scaler fit on entire dataset:')
print('  mean_:', np.round(scaler_wrong.mean_, 6))
print('  scale_:', np.round(scaler_wrong.scale_, 6))

WRONG order — scaler fit on entire dataset:
  mean_: [19.561667 65.922017  2.165   ]
  scale_: [17.524541 22.799319  1.498146]


In [8]:
# Step 2 (CORRECT order): split FIRST, fit scaler on training split ONLY
X_train_num, X_test_num, y_train_num, y_test_num = train_test_split(
    X_num, y_all, test_size=0.25, random_state=RANDOM_STATE, stratify=y_all
)
scaler_right = StandardScaler()
scaler_right.fit(X_train_num)  # train only — no test contamination
print('CORRECT order — scaler fit on training split only:')
print('  mean_:', np.round(scaler_right.mean_, 6))
print('  scale_:', np.round(scaler_right.scale_, 6))

CORRECT order — scaler fit on training split only:
  mean_: [19.895556 65.794733  2.117778]
  scale_: [17.445158 22.263904  1.483807]


In [9]:
# Step 3: numeric difference between the two scalers' means
mean_diff = scaler_wrong.mean_ - scaler_right.mean_
print('Difference in mean_ (wrong - right):', np.round(mean_diff, 6))
print('Max absolute difference:', f'{np.abs(mean_diff).max():.6f}')

Difference in mean_ (wrong - right): [-0.333889  0.127283  0.047222]
Max absolute difference: 0.333889


**Why the principle matters regardless of gap size:** Even if today's gap looks tiny, the wrong order leaks information from the test set into every training run — the scaler's `mean_` and `scale_` encode the test distribution, so the model indirectly trains on data it should never see. On a different split, a smaller dataset, or data that shifts over time, that gap can grow arbitrarily large and silently inflate validation scores. The only way to guarantee an honest estimate of production performance is to make the correct order a structural habit (split first, fit preprocessing on train only), not a judgement call about whether the numbers 'look fine' this time.

In [10]:
# Step 4: rebuild correctly as a single sklearn.Pipeline (imputer + scaler + model)
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
pipe.fit(X_train_num, y_train_num)  # fit only on the training split

pipe_scaler = pipe.named_steps['scaler']
print('Pipeline scaler mean_:  ', np.round(pipe_scaler.mean_, 6))
print('Manual scaler mean_:    ', np.round(scaler_right.mean_, 6))
print('Exact match (mean_):', np.allclose(pipe_scaler.mean_, scaler_right.mean_, atol=1e-12))
print()
print('Pipeline scaler scale_: ', np.round(pipe_scaler.scale_, 6))
print('Manual scaler scale_:   ', np.round(scaler_right.scale_, 6))
print('Exact match (scale_):', np.allclose(pipe_scaler.scale_, scaler_right.scale_, atol=1e-12))

Pipeline scaler mean_:   [19.895556 65.794733  2.117778]
Manual scaler mean_:     [19.895556 65.794733  2.117778]
Exact match (mean_): True

Pipeline scaler scale_:  [17.445158 22.263904  1.483807]
Manual scaler scale_:    [17.445158 22.263904  1.483807]
Exact match (scale_): True


## Section 4.3 — The fix

Retrain the final model using only legitimate features and a properly ordered pipeline (split first, fit preprocessing on train only). This should match Model A from Section 4.1 — the honest, deployable version.

In [11]:
# Final fixed model: legitimate features + correctly ordered pipeline
# Reuse the exact same train/test split from Section 4.1 for a fair comparison
numeric_features = ['tenure_months', 'monthly_charges', 'support_calls']
categorical_features = ['contract_type']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

final_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

# X_train / X_test from 4.1 contain legit + leaked cols; select legit only
final_pipe.fit(X_train[legit_features], y_train)
y_pred_final = final_pipe.predict(X_test[legit_features])
y_proba_final = final_pipe.predict_proba(X_test[legit_features])[:, 1]

acc_final = accuracy_score(y_test, y_pred_final)
auc_final = roc_auc_score(y_test, y_proba_final)

print(f'Final fixed model  accuracy: {acc_final:.4f}   ROC-AUC: {auc_final:.4f}')
print(f'Model A (baseline) accuracy: {acc_a:.4f}   ROC-AUC: {auc_a:.4f}')
print(f'Accuracy match: {np.isclose(acc_final, acc_a, atol=1e-9)}')
print(f'ROC-AUC  match: {np.isclose(auc_final, auc_a, atol=1e-9)}')

Final fixed model  accuracy: 0.7533   ROC-AUC: 0.7783
Model A (baseline) accuracy: 0.7533   ROC-AUC: 0.7783
Accuracy match: True
ROC-AUC  match: True


**Confirmation:** The final fixed model's accuracy and ROC-AUC match Model A from Section 4.1 exactly — same legitimate features, same train/test split, same correctly ordered preprocessing (fit on train only). It is the honest, deployable version: no target-leaked columns, no scaler that peeked at the test set.

## Bonus — Cross-validation can leak too

In [12]:
# LEAKY: fit scaler on the WHOLE dataset, then run CV on pre-scaled data
scaler_cv_leak = StandardScaler()
X_scaled_leak = scaler_cv_leak.fit_transform(X_num)  # saw every row, including all held-out folds

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
scores_leaky = cross_val_score(lr, X_scaled_leak, y_all, cv=5, scoring='accuracy')

# CORRECT: scaler INSIDE the Pipeline passed to cross_val_score
# each fold's scaler is fit only on that fold's training rows
pipe_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
scores_correct = cross_val_score(pipe_cv, X_num, y_all, cv=5, scoring='accuracy')

print('Leaky CV   scores:', np.round(scores_leaky, 4), f'  mean={scores_leaky.mean():.4f}')
print('Correct CV scores:', np.round(scores_correct, 4), f'  mean={scores_correct.mean():.4f}')
print(f'Mean difference (leaky - correct): {scores_leaky.mean() - scores_correct.mean():+.4f}')

Leaky CV   scores: [0.75   0.75   0.7583 0.7417 0.7417]   mean=0.7483
Correct CV scores: [0.75   0.75   0.7583 0.7417 0.7417]   mean=0.7483
Mean difference (leaky - correct): +0.0000


**Explanation:**

When the scaler is fit on the whole dataset before CV, every held-out fold's statistics (mean, scale) leak into the training folds through the transformed features, so each fold's model has indirectly seen distributional information from the rows it's being evaluated on — this can inflate CV scores relative to the properly wrapped Pipeline, where each fold's scaler is fit only on that fold's training rows. On this particular dataset the difference may be small (and can even occasionally go the other way by random chance) because the numeric features are roughly stable across folds and `StandardScaler` is a simple, per-feature linear transform — the leakage is subtle here. But the risk is still real in general: with fewer rows, heavier skew, outliers concentrated in one fold, or leakier transformations (e.g., target encoding, feature selection, iterative imputation), pre-CV fitting can meaningfully and silently overstate performance. Wrapping preprocessing inside the Pipeline passed to `cross_val_score` is the only structure that guarantees each evaluation fold is truly unseen.